# Bootstrap Resampling: From General Statistics to Paleomagnetism

The bootstrap (Efron, 1979) estimates uncertainty in virtually any statistic — means, medians, regression coefficients, eigenvalues — without specifying a parametric distribution such as Gaussian or Fisher. This makes it especially powerful for Earth science data, which can often deviate from idealized distributional assumptions.

This notebook develops the bootstrap from the ground up:

1. **Why the bootstrap works** — the plug-in principle
2. **How it works** — resampling with replacement
3. **Application to directional data** — bootstrap confidence regions for paleomagnetic means
4. **The bootstrap fold test** — Tauxe and Watson (1994)

### References

- Efron, B. (1979). Bootstrap methods: another look at the jackknife. *Annals of Statistics*, 7, 1–26.
- Efron, B. and Tibshirani, R. (1993). *An Introduction to the Bootstrap*. Chapman and Hall.
- Tauxe, L., Kylstra, N., and Constable, C. (1991). Bootstrap statistics for paleomagnetic data. *J. Geophys. Res.*, 96, 11,723–11,740. doi:[10.1029/91JB00572](https://doi.org/10.1029/91JB00572)
- Tauxe, L. and Watson, G. S. (1994). The fold test: an eigen analysis approach. *Earth Planet. Sci. Lett.*, 122, 331–341. doi:[10.1016/0012-821X(94)90006-X](https://doi.org/10.1016/0012-821X(94)90006-X)

In [ ]:
!pip install pmagpy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pmagpy.ipmag as ipmag
import pmagpy.pmag as pmag
from ipywidgets import interact, IntSlider

%matplotlib inline
%config InlineBackend.figure_format='retina'

## Part 1: Why does the bootstrap work?

### The fundamental problem

We have a *sample* from some *population* and want to estimate a parameter (like the mean) and its uncertainty. Classical parametric statistics assumes the data follow a known distribution family (e.g., Fisher) and derives formulas accordingly. When that assumption fails due to complexities such as bimodal data, outliers, or elliptical distributions, the formulas can mislead.

### The plug-in principle

The bootstrap rests on a simple idea:

> *The empirical distribution of the sample is the best available approximation to the true population distribution.*

The true population distribution $F$ is unknown. The **empirical distribution** $\hat{F}$ puts probability $1/n$ on each of the $n$ observed data points. To learn how a statistic varies across repeated sampling from $F$, we study how it varies across repeated sampling from $\hat{F}$.

Sampling from $\hat{F}$ is equivalent to **drawing $n$ values from your data with replacement**. Each such draw is a **bootstrap pseudo-sample**. Some observations appear multiple times; others are absent.

The algorithm:

1. Draw $n$ values from your data **with replacement**
2. Compute the statistic of interest from the pseudo-sample
3. Repeat many times (hundreds to thousands)
4. The distribution of the statistic across pseudo-samples approximates its true sampling distribution

### Why this works

| Real world | Bootstrap world |
|---|---|
| Population $F$ | Sample (empirical distribution $\hat{F}$) |
| Sample drawn from $F$ | Pseudo-sample drawn from $\hat{F}$ |
| Statistic from sample | Statistic from pseudo-sample |
| Sampling distribution | Bootstrap distribution |

For large enough samples, $\hat{F}$ converges to $F$ (Glivenko-Cantelli theorem). For sufficiently representative samples and well-behaved statistics, the bootstrap distribution approximates the true sampling distribution well — though performance depends on the statistic, sample size, and data characteristics.

### Broad applicability

The same resampling procedure works for means, medians, regression coefficients, correlation coefficients, eigenvalues — any statistic you can compute from data. No new theory is needed for each case. This generality is why the bootstrap has become ubiquitous across fields from medicine to economics to Earth science.

## Part 2: The bootstrap in action — a simple example

We simulate measuring the long-axis lengths of 20 clasts from a conglomerate. Clast size distributions are commonly log-normal, so we draw from that population and compare the bootstrap to the known truth.

In [ ]:
rng = np.random.default_rng(42)

# simulate 20 clast size measurements from a log-normal distribution
n = 20
clast_sizes = rng.lognormal(mean=1.0, sigma=0.5, size=n)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(clast_sizes, bins=10, edgecolor='black', alpha=0.7, color='steelblue')
ax.axvline(clast_sizes.mean(), color='red', linewidth=2, label=f'Sample mean = {clast_sizes.mean():.2f} cm')
ax.set_xlabel('Clast long-axis length (cm)')
ax.set_ylabel('Count')
ax.set_title(f'20 simulated clast size measurements')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Sample size: {n}')
print(f'Sample mean: {clast_sizes.mean():.2f} cm')
print(f'Sample median: {np.median(clast_sizes):.2f} cm')

### One bootstrap pseudo-sample

We draw 20 values from our 20 measurements *with replacement*. Some get picked multiple times, others are left out.

In [ ]:
# draw one bootstrap pseudo-sample
indices = rng.integers(0, n, size=n)
pseudo_sample = clast_sizes[indices]

print("Original data (sorted):")
print(np.sort(clast_sizes).round(2))
print()
print("Indices drawn (with replacement):")
print(indices)
print()
print("Bootstrap pseudo-sample (sorted):")
print(np.sort(pseudo_sample).round(2))
print()
print(f"Original mean:        {clast_sizes.mean():.3f} cm")
print(f"Pseudo-sample mean:   {pseudo_sample.mean():.3f} cm")
print()

# which observations were selected and how many times?
unique, counts = np.unique(indices, return_counts=True)
print(f"Observations appearing 0 times: {n - len(unique)}")
print(f"Observations appearing 2+ times: {(counts > 1).sum()}")

### Many bootstrap replicates

Each pseudo-sample gives a different mean. The spread of these means tells us how much the sample mean would vary across repeated experiments. Let's generate 5,000 pseudo-samples and examine the **bootstrap distribution of the mean**.

In [ ]:
num_boot = 5000
boot_means = np.zeros(num_boot)

for i in range(num_boot):
    idx = rng.integers(0, n, size=n)
    boot_means[i] = clast_sizes[idx].mean()

# 95% confidence interval from the bootstrap distribution
ci_lower = np.percentile(boot_means, 2.5)
ci_upper = np.percentile(boot_means, 97.5)

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(boot_means, bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax.axvline(clast_sizes.mean(), color='red', linewidth=2, label=f'Sample mean = {clast_sizes.mean():.2f}')
ax.axvline(ci_lower, color='orange', linewidth=2, linestyle='--', label=f'95% CI: [{ci_lower:.2f}, {ci_upper:.2f}]')
ax.axvline(ci_upper, color='orange', linewidth=2, linestyle='--')
ax.set_xlabel('Bootstrap mean clast size (cm)')
ax.set_ylabel('Density')
ax.set_title(f'Bootstrap distribution of the mean ({num_boot} replicates)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Bootstrap estimate of the standard error of the mean: {boot_means.std():.3f} cm")
print(f"Bootstrap 95% confidence interval: [{ci_lower:.2f}, {ci_upper:.2f}] cm")

### Any statistic works

The same procedure applies to *any* statistic. Classical formulas for median confidence intervals are awkward; the bootstrap handles mean and median identically.

In [ ]:
boot_medians = np.zeros(num_boot)

for i in range(num_boot):
    idx = rng.integers(0, n, size=n)
    boot_medians[i] = np.median(clast_sizes[idx])

ci_mean = np.percentile(boot_means, [2.5, 97.5])
ci_median = np.percentile(boot_medians, [2.5, 97.5])

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

axes[0].hist(boot_means, bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True)
axes[0].axvline(ci_mean[0], color='orange', linewidth=2, linestyle='--')
axes[0].axvline(ci_mean[1], color='orange', linewidth=2, linestyle='--')
axes[0].axvline(clast_sizes.mean(), color='red', linewidth=2)
axes[0].set_xlabel('Bootstrap mean (cm)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Mean: [{ci_mean[0]:.2f}, {ci_mean[1]:.2f}]')

axes[1].hist(boot_medians, bins=50, edgecolor='black', alpha=0.7, color='seagreen', density=True)
axes[1].axvline(ci_median[0], color='orange', linewidth=2, linestyle='--')
axes[1].axvline(ci_median[1], color='orange', linewidth=2, linestyle='--')
axes[1].axvline(np.median(clast_sizes), color='red', linewidth=2)
axes[1].set_xlabel('Bootstrap median (cm)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Median: [{ci_median[0]:.2f}, {ci_median[1]:.2f}]')

plt.suptitle('Bootstrap 95% confidence intervals', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

### Verification against the known truth

Because we simulated from a known distribution, we know the true population mean. Does the bootstrap confidence interval contain it?

In [ ]:
# the true population mean of a log-normal with parameters mu=1.0, sigma=0.5
true_mean = np.exp(1.0 + 0.5**2 / 2)
boot_ci = np.percentile(boot_means, [2.5, 97.5])

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(boot_means, bins=50, edgecolor='black', alpha=0.7, color='steelblue', density=True)
ax.axvline(clast_sizes.mean(), color='red', linewidth=2, label=f'Sample mean = {clast_sizes.mean():.2f}')
ax.axvline(true_mean, color='black', linewidth=2, linestyle=':', label=f'True population mean = {true_mean:.2f}')
ax.axvline(boot_ci[0], color='orange', linewidth=2, linestyle='--', label=f'95% CI: [{boot_ci[0]:.2f}, {boot_ci[1]:.2f}]')
ax.axvline(boot_ci[1], color='orange', linewidth=2, linestyle='--')
ax.set_xlabel('Bootstrap mean clast size (cm)')
ax.set_ylabel('Density')
ax.set_title('Bootstrap distribution of the mean vs. true population mean')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

contains = boot_ci[0] <= true_mean <= boot_ci[1]
print(f"True population mean: {true_mean:.2f} cm")
print(f"Bootstrap 95% CI:     [{boot_ci[0]:.2f}, {boot_ci[1]:.2f}] cm")
print(f"CI contains true mean: {contains}")

## Part 3: Bootstrap resampling for directional data

Paleomagnetic directions live on the unit sphere, which creates challenges for classical statistics:

1. **Fisher statistics assume spherical symmetry.** Real paleomagnetic data are often elliptically distributed, bimodal, or contain intermediate directions.
2. **The Fisher $\alpha_{95}$ cone** is tied to a circular confidence geometry under the Fisher distribution. When directional uncertainty is elliptical or the data are non-Fisherian, this can be a poor description of the actual confidence region.
3. **Distributions are distorted when converting from directions to poles (or vice versa)** as this geometric transformation does not preserve circular symmetry

The bootstrap provides confidence regions that naturally accommodate elliptical scatter and require no assumption of a particular parametric distribution (Tauxe et al., 1991).

In PmagPy, `pmag.pseudo()` draws a bootstrap pseudo-sample of [dec, inc] pairs — the same resampling-with-replacement operation applied to directional data:

```python
def pseudo(DIs, random_seed=None):
    rng = _resolve_rng(random_seed)
    Inds = rng.integers(len(DIs), size=len(DIs))
    return np.array(DIs)[Inds]
```

`pmag.di_boot()` wraps this to generate many bootstrap mean directions. Let's see it work on simulated Fisher-distributed data.

In [ ]:
# generate 30 directions from a Fisher distribution
# true mean: dec=0, inc=60 (like a mid-latitude Northern Hemisphere site)
# kappa=50 (moderately well-grouped)
n_dirs = 30
kappa = 50
true_dec, true_inc = 0., 60.

di_block = ipmag.fishrot(n = n_dirs, dec = true_dec, inc = true_inc, k=kappa)
fpars = ipmag.fisher_mean(di_block[:, 0].tolist(), di_block[:, 1].tolist())

print(f"Fisher mean: Dec = {fpars['dec']:.1f}°, Inc = {fpars['inc']:.1f}°")
print(f"n = {fpars['n']}, k = {fpars['k']:.1f}, α95 = {fpars['alpha95']:.1f}°")

# plot the data
fig, ax = plt.subplots(1, 1, figsize=(4.5, 4.5))
plt.sca(ax)
ipmag.plot_net()
ipmag.plot_di(di_block[:, 0], di_block[:, 1], color='k', markersize=30)
ipmag.plot_di(fpars['dec'], fpars['inc'], color='red', markersize=80, marker='*')
plt.title(f"Simulated Fisher directions (n={n_dirs}, κ={kappa})")
plt.tight_layout()
plt.show()

### Bootstrap pseudo-samples of directions

Each pseudo-sample has the same number of directions as the original, but with different random subsets due to resampling with replacement. The mean (red star) shifts slightly each time.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# original data
plt.sca(axes[0])
ipmag.plot_net()
ipmag.plot_di(di_block[:, 0], di_block[:, 1], color='k', markersize=30)
ipmag.plot_di(fpars['dec'], fpars['inc'], color='red', markersize=80, marker='*')
plt.title('Original data')

# three bootstrap pseudo-samples
for j in range(1, 4):
    plt.sca(axes[j])
    ipmag.plot_net()
    pseudo = pmag.pseudo(di_block.tolist(), random_seed=rng)
    pseudo = np.array(pseudo)
    pfpars = ipmag.fisher_mean(pseudo[:, 0].tolist(), pseudo[:, 1].tolist())
    ipmag.plot_di(pseudo[:, 0], pseudo[:, 1], color='steelblue', markersize=30)
    ipmag.plot_di(pfpars['dec'], pfpars['inc'], color='red', markersize=80, marker='*')
    plt.title(f'Pseudo-sample {j}')

plt.suptitle('Original data and three bootstrap pseudo-samples (means shown as red stars)', y=1.02)
plt.tight_layout()
plt.show()

### Bootstrap confidence region vs. Fisher $\alpha_{95}$

The scatter of bootstrap means across many replicates defines the confidence region. We compare it to the Fisher $\alpha_{95}$ circle.

In [ ]:
# generate 500 bootstrap mean directions using PmagPy's di_boot
boot_means_di = pmag.di_boot(di_block.tolist(), nb=500, random_seed=42)
boot_means_di = np.array(boot_means_di)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# left: data with α95 circle
plt.sca(axes[0])
ipmag.plot_net()
ipmag.plot_di(di_block[:, 0], di_block[:, 1], color='k', markersize=30)
ipmag.plot_di(fpars['dec'], fpars['inc'], color='red', markersize=80, marker='*')
a95_circle_decs, a95_circle_incs = [], []
for angle in np.linspace(0, 360, 361):
    d, i = pmag.dodirot(angle, 90 - fpars['alpha95'], fpars['dec'], fpars['inc'])
    a95_circle_decs.append(d)
    a95_circle_incs.append(i)
ipmag.plot_di(a95_circle_decs, a95_circle_incs, color='red', markersize=1)
plt.title(f"Data with Fisher α95 = {fpars['alpha95']:.1f}°")

# right: bootstrap means
plt.sca(axes[1])
ipmag.plot_net()
ipmag.plot_di(boot_means_di[:, 0], boot_means_di[:, 1], color='steelblue', markersize=10)
ipmag.plot_di(fpars['dec'], fpars['inc'], color='red', markersize=80, marker='*')
ipmag.plot_di(a95_circle_decs, a95_circle_incs, color='red', markersize=1)
plt.title('500 bootstrap mean directions')

plt.suptitle('Fisher α95 (red circle) vs. bootstrap confidence region (blue cloud)', y=1.02)
plt.tight_layout()
plt.show()

### Cartesian component view

Following Tauxe et al. (1991), we can convert bootstrap means to Cartesian coordinates (x, y, z) and plot histograms. Unimodal component histograms are consistent with a single cluster and provide a useful visual diagnostic of the bootstrap structure. This view is central to the common mean test and fold test.

In [ ]:
# convert bootstrap means to Cartesian coordinates
boot_cart = np.array([pmag.dir2cart([d[0], d[1], 1.0]) for d in boot_means_di])

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
labels = ['X (North)', 'Y (East)', 'Z (Down)']
colors = ['steelblue', 'seagreen', 'coral']

for i, (ax, label, color) in enumerate(zip(axes, labels, colors)):
    ax.hist(boot_cart[:, i], bins=40, edgecolor='black', alpha=0.7, color=color)
    ax.axvline(boot_cart[:, i].mean(), color='red', linewidth=2)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')

plt.suptitle('Cartesian components of 500 bootstrap mean directions', y=1.02)
plt.tight_layout()
plt.show()

## Part 4: The bootstrap fold test

The fold test asks: was the magnetization acquired *before* or *after* tilting? If pre-folding, directions cluster more tightly after tilt correction. The classic McElhinny (1964) test compared $\kappa$ before and after correction, but the two estimates of $\kappa$ are not independent, the data need not be Fisher distributed, and the test cannot detect intermediate (syn-folding) remanence.

### The Tauxe and Watson (1994) approach

The key innovation is to use the **largest eigenvalue ($\tau_1$) of the orientation matrix** as the clustering measure:

$$\mathbf{T} = \frac{1}{n} \mathbf{X}^T \mathbf{X}$$

The eigenvalues $\tau_1 \geq \tau_2 \geq \tau_3$ sum to 1. Tightly clustered data have $\tau_1 \approx 1$. Unlike $\kappa$, the orientation matrix is polarity-independent and requires no assumption of Fisher symmetry.

### The algorithm

1. Draw a bootstrap pseudo-sample (directions + bedding orientations)
2. Apply incremental untilting (e.g., -10% to 150%) and compute $\tau_1$ at each step
3. Record the % untilting that maximizes $\tau_1$
4. Repeat for many pseudo-samples
5. The 95% bounds on the $\tau_1$-maximizing percentages give the confidence interval

**Interpretation:**
- 95% bounds include 100% → **positive** (pre-folding)
- 95% bounds include 0% → **negative** (post-folding)
- 95% bounds exclude both → best clustering at intermediate unfolding, **commonly interpreted as syn-folding** — though structural complexity, vertical-axis rotations, or mixed magnetization components can produce intermediate maxima as well

### Building intuition: $\tau_1$ vs. percent unfolding

First, let's see how $\tau_1$ varies with unfolding for a synthetic pre-folding data set (Fisher-distributed directions "folded" with two limbs).

In [ ]:
# create a synthetic pre-folding data set
# start with tightly grouped directions, then apply two different tilts
rng_fold = np.random.default_rng(123)
n_per_limb = 15
kappa_fold = 40

# generate tilt-corrected directions (the "true" pre-folding magnetization)
true_directions = []
for i in range(n_per_limb * 2):
    d, inc = pmag.fshdev(kappa_fold, random_seed=rng_fold)
    drot, irot = pmag.dodirot(d, inc, 350., 55.)
    true_directions.append([drot, irot])
true_directions = np.array(true_directions)

# assign bedding attitudes: two "limbs" of a fold
# limb 1: dip_direction=90, dip=30 (tilted to the east)
# limb 2: dip_direction=270, dip=40 (tilted to the west)
bedding = np.zeros((n_per_limb * 2, 2))
bedding[:n_per_limb] = [90., 30.]
bedding[n_per_limb:] = [270., 40.]

# "fold" the data: apply the bedding tilt to get geographic coordinates
geo_directions = np.zeros_like(true_directions)
for i in range(len(true_directions)):
    d, inc = pmag.dotilt(true_directions[i, 0], true_directions[i, 1],
                          bedding[i, 0], -bedding[i, 1])
    geo_directions[i] = [d, inc]

# build the DIDDD array [dec, inc, dip_direction, dip] in geographic coordinates
diddd = np.column_stack([geo_directions, bedding])

# plot geographic and tilt-corrected directions
D_tc, I_tc = pmag.dotilt_V(diddd)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

plt.sca(axes[0])
ipmag.plot_net()
ipmag.plot_di(diddd[:n_per_limb, 0], diddd[:n_per_limb, 1], color='steelblue', markersize=40)
ipmag.plot_di(diddd[n_per_limb:, 0], diddd[n_per_limb:, 1], color='coral', markersize=40)
plt.title('Geographic (in situ)')

plt.sca(axes[1])
ipmag.plot_net()
ipmag.plot_di(D_tc[:n_per_limb], I_tc[:n_per_limb], color='steelblue', markersize=40)
ipmag.plot_di(D_tc[n_per_limb:], I_tc[n_per_limb:], color='coral', markersize=40)
plt.title('Tilt-corrected')

plt.suptitle('Blue = Limb 1 (dip 30°E), Orange = Limb 2 (dip 40°W)', y=1.02)
plt.tight_layout()
plt.show()

The two limbs separate in geographic coordinates but converge after tilt correction. Use the slider below to explore what happens at intermediate degrees of unfolding:

In [ ]:
@interact(
    pct_unfold=IntSlider(value=0, min=-20, max=140, step=5, description='% Unfolding')
)
def interactive_unfold(pct_unfold):
    tilt = np.array([1., 1., 1., 0.01 * pct_unfold])
    D_part, I_part = pmag.dotilt_V(diddd * tilt)
    ppars = pmag.doprinc(np.column_stack([D_part, I_part]))

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    # stereonet
    plt.sca(axes[0])
    ipmag.plot_net()
    ipmag.plot_di(D_part[:n_per_limb], I_part[:n_per_limb], color='steelblue', markersize=40)
    ipmag.plot_di(D_part[n_per_limb:], I_part[n_per_limb:], color='coral', markersize=40)
    plt.title(f'{pct_unfold}% unfolding')

    # tau_1 curve with current position marked
    percs_all = np.arange(-20, 141)
    taus_all = []
    for p in percs_all:
        t = np.array([1., 1., 1., 0.01 * p])
        Dp, Ip = pmag.dotilt_V(diddd * t)
        pp = pmag.doprinc(np.column_stack([Dp, Ip]))
        taus_all.append(pp['tau1'])

    axes[1].plot(percs_all, taus_all, 'k-', linewidth=2)
    axes[1].axvline(pct_unfold, color='red', linewidth=2, linestyle='-')
    axes[1].plot(pct_unfold, ppars['tau1'], 'ro', markersize=10)
    axes[1].set_xlabel('% Unfolding')
    axes[1].set_ylabel('τ₁')
    axes[1].set_title(f'τ₁ = {ppars["tau1"]:.4f}')

    plt.tight_layout()
    plt.show()

In [ ]:
# compute tau_1 as a function of percent unfolding
percs = np.arange(-20, 141)
taus = []

for perc in percs:
    tilt = np.array([1., 1., 1., 0.01 * perc])
    D, I = pmag.dotilt_V(diddd * tilt)
    TCs = np.array([D, I]).transpose()
    ppars = pmag.doprinc(TCs)
    taus.append(ppars['tau1'])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(percs, taus, 'k-', linewidth=2)
ax.axvline(100, color='gray', linestyle=':', linewidth=1, label='100% unfolding')
ax.axvline(0, color='gray', linestyle='--', linewidth=1, label='0% (geographic)')
ax.set_xlabel('% Unfolding')
ax.set_ylabel('τ₁ (largest eigenvalue)')
ax.set_title('Concentration of directions as a function of unfolding')
ax.legend()
plt.tight_layout()
plt.show()

max_idx = np.argmax(taus)
print(f"Maximum τ₁ = {taus[max_idx]:.4f} at {percs[max_idx]}% unfolding")

The peak is near 100% unfolding, but how certain are we? Each bootstrap pseudo-sample produces a slightly different curve with a slightly different peak. Tracking these peaks builds the confidence interval.

### The bootstrap fold test step by step

This code does exactly what `ipmag.bootstrap_fold_test()` does internally.

In [ ]:
# manually run the bootstrap fold test on our synthetic data
# to show step-by-step what ipmag.bootstrap_fold_test() does internally
num_sims = 100
percs = list(range(-20, 141))
untilt_maxima = []

fig, ax = plt.subplots(figsize=(8, 4.5))

for n_sim in range(num_sims):
    taus_boot = []
    # step 1: draw a pseudo-sample
    PDs = pmag.pseudo(diddd, random_seed=rng_fold)

    # step 2: compute tau_1 at each unfolding step
    for perc in percs:
        tilt = np.array([1., 1., 1., 0.01 * perc])
        D, I = pmag.dotilt_V(PDs * tilt)
        TCs = np.array([D, I]).transpose()
        ppars = pmag.doprinc(TCs)
        taus_boot.append(ppars['tau1'])

    # plot the first 25 curves
    if n_sim < 25:
        ax.plot(percs, taus_boot, 'r--', alpha=0.3, linewidth=0.8)

    # step 3: record where tau_1 is maximized
    untilt_maxima.append(percs[np.argmax(taus_boot)])

# plot the last curve in black for reference
ax.plot(percs, taus_boot, 'k-', linewidth=1.5)

# overlay the CDF
untilt_maxima.sort()
cdf = np.arange(num_sims) / num_sims
ax.plot(untilt_maxima, cdf, 'g-', linewidth=2)

# confidence bounds
lower = int(0.025 * num_sims)
upper = int(0.975 * num_sims)
ax.axvline(untilt_maxima[lower], color='green', linestyle='--', linewidth=1.5)
ax.axvline(untilt_maxima[upper], color='green', linestyle='--', linewidth=1.5)

ax.set_xlabel('% Unfolding')
ax.set_ylabel('τ₁ (red dashed) / CDF (green)')
ax.set_title(f'Bootstrap fold test: 95% CI = [{untilt_maxima[lower]}, {untilt_maxima[upper]}]% unfolding')
plt.tight_layout()
plt.show()

print(f"95% confidence bounds: {untilt_maxima[lower]}% to {untilt_maxima[upper]}% unfolding")
if untilt_maxima[lower] <= 100 <= untilt_maxima[upper]:
    print("→ 100% unfolding is within the 95% bounds: PASSES the fold test")
    print("  (consistent with a pre-folding magnetization)")
else:
    print("→ 100% unfolding is NOT within the 95% bounds")

### Reading the plot

- **Red dashed curves**: $\tau_1$ vs. % unfolding for individual pseudo-samples
- **Green curve**: CDF of where $\tau_1$ peaks across all pseudo-samples
- **Green dashed verticals**: 95% confidence bounds (2.5th and 97.5th percentiles)

### Using `ipmag.bootstrap_fold_test()`

PmagPy encapsulates this procedure. It takes a numpy array of `[dec, inc, dip_direction, dip]` in geographic coordinates and produces the three standard plots.

In [ ]:
ipmag.bootstrap_fold_test(diddd, num_sims=500, min_untilt=-20, max_untilt=140, random_seed=42)

## Part 5: A post-folding magnetization example

For contrast, here is a synthetic data set where the magnetization was acquired *after* folding (a remagnetization). The 95% bounds should center near 0% unfolding.

In [ ]:
# post-folding magnetization: generate directions in geographic coordinates
# (the magnetization was acquired after tilting, so geographic = the true reference frame)
rng_post = np.random.default_rng(456)

post_geo = []
for i in range(n_per_limb * 2):
    d, inc = pmag.fshdev(kappa_fold, random_seed=rng_post)
    drot, irot = pmag.dodirot(d, inc, 350., 55.)
    post_geo.append([drot, irot])
post_geo = np.array(post_geo)

# use the same bedding attitudes as before
diddd_post = np.column_stack([post_geo, bedding])

print("Running bootstrap fold test on post-folding (remagnetized) synthetic data...")
print("The 95% bounds should be near 0% unfolding.\n")
ipmag.bootstrap_fold_test(diddd_post, num_sims=100, min_untilt=-20, max_untilt=140, random_seed=42)

The 95% bounds center near 0% unfolding — tilt correction increases scatter rather than reducing it, as expected for a post-folding magnetization.

## Summary

The bootstrap estimates the sampling distribution of a statistic by resampling with replacement from the observed data. It works because the empirical distribution is the best available approximation to the true population, and each pseudo-sample simulates a plausible re-run of the experiment.

For paleomagnetism, the bootstrap provides confidence regions that are distribution-free and naturally elliptical (Tauxe et al., 1991) — a significant improvement over Fisher $\alpha_{95}$ circles for non-ideal data. The bootstrap fold test (Tauxe and Watson, 1994) applies the same principle to the eigenvalue $\tau_1$ as a function of unfolding, providing continuous confidence bounds on whether the magnetization is pre-folding, post-folding, or syn-folding.

The key insight throughout: the bootstrap fold test is not a special-purpose algorithm — it is the same resampling logic that gave us confidence intervals for clast sizes, now applied to $\tau_1$ vs. unfolding percentage.